# Reverse MicrobioLink: Bacterial Motifs Engaging Human Domains

This notebook walks through the **reverse MicrobioLink pipeline** step by step on a
small mocked dataset.

## What the reverse pipeline asks

The *forward* MicrobioLink pipeline looks for ELM short linear motifs (SLiMs) in
**human** proteins and asks: which bacterial proteins carry a Pfam domain that
recognises one of those motifs?

The **reverse** pipeline flips the question: which **bacterial** proteins carry an ELM
motif, and which **human** proteins expose a Pfam domain that could recognise it?
This addresses the phenomenon of *molecular mimicry*, where pathogens evolve short
sequence elements that resemble host SLiMs, enabling them to co-opt host-cell
signalling machinery.

## Three-stage matching algorithm

1. **Scan** bacterial protein sequences for ELM motif patterns (regex matching).
2. **Look up** which Pfam domains each matched motif interacts with (ELM interaction
   database).
3. **Look up** which human proteins carry those Pfam domains (UniProt annotation
   table).

The result is a table linking a bacterial protein and its embedded motif to a human
protein domain that could be engaged by that motif.

## An important pre-processing step: removing CLV_ motifs

The ELM database classifies motifs into functional categories.  The `CLV_` prefix
marks *cleavage site* motifs — sequence patterns that are recognised by proteases and
result in the protein being cut, not bound.  These are fundamentally different from
domain-binding motifs such as `LIG_` or `DEG_`, and including them would generate
false predictions about domain-binding interactions.  We therefore filter them out
before scanning.

## This example is intentionally minimal

- The bacterial FASTA is mocked inline.
- The human domain table is mocked inline (real pipelines feed the output of
  `download_human_domains` here).
- The ELM motif regex and motif-domain interaction tables are the real packaged
  resources that ship with `microbiolink_api`.

In [3]:
# Would not want this for a final tutorial, very verbose and not very user friendly

from pathlib import Path
import shutil
import subprocess
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists() and (path / 'microbiolink').exists():
            return path
    raise RuntimeError('Could not find the MicrobioLink repository root.')


repo_root = find_repo_root()
workdir = repo_root / 'tutorials' / 'mock_data' / 'runs' / '03_mock_reverse_dmi'

if workdir.exists():
    shutil.rmtree(workdir)

workdir.mkdir(parents=True)
repo_root, workdir

(PosixPath('/Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2'),
 PosixPath('/Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2/tutorials/mock_data/runs/03_mock_reverse_dmi'))

In [ ]:
### In a real tutorial, we should make sure that the package here is installed in pypi

subprocess.run(
    ['uv', 'pip', 'install', '-e', str(repo_root)],
    check=True,
    cwd=repo_root,
)

Audited 1 package in 575ms


CompletedProcess(args=['uv', 'pip', 'install', '-e', '/Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2'], returncode=0)

## Import the required packages for running Reverse MicrobioLINK

In [1]:
import dataclasses

import pandas as pd

from microbiolink_api import (
    extract_uniprot_id,
    load_default_dmi_resource_bundle,
    read_bacterial_domain_table,
    read_fasta_sequences,
)
from microbiolink.reverse_DMI import (
    filter_cleavage_motifs,
    predict_reverse_domain_motif_interactions_from_data,
)

## Step 1: Create mock inputs

### Bacterial FASTA

We create two mock bacterial proteins.

**`A0A001` — `DEG_BACT` (positive control)**

This protein begins with the tripeptide `MCR`.  The ELM database defines a class of
N-degron motifs recognised by the UBR-box domain of N-recognin E3 ubiquitin ligases.
One of these classes, `DEG_Nend_UBRbox_4`, has the regex `^M{0,1}(C).`: an
optional N-terminal methionine followed immediately by a cysteine and one more
residue.  The sequence `MCR` matches this pattern at position 0.  This is the motif
we expect the pipeline to detect.

In a real analysis this kind of sequence would represent a bacterial protein whose
N-terminal region has evolved to resemble a eukaryotic degradation signal, potentially
allowing it to engage the host ubiquitin pathway.

**`A0A002` — `CLV_BACT` (negative / filtering control)**

This protein contains the subsequence `RRKRG` (embedded at positions 3–8 of the
sequence `GGGRRRKRG...`).  The substring matches the furin cleavage site regex
`R.[RK]R.` (`CLV_PCSK_FUR_1`): R, any residue, R or K, R, any residue.  Furin is a
proprotein convertase that *cuts* its substrate at this site — it does not *bind* the
motif like a domain-mediated interaction.  Because `CLV_` motifs are removed before
scanning (see Step 2), `A0A002` will not appear in the final output.

Note that the sequence begins with `G` (glycine), so none of the N-degron patterns
(which require `^M{0,1}[...]`) can match.  Combined with the CLV filter, `A0A002`
produces zero output rows — a clean demonstration of the filtering step.

### Human domain table

We create a minimal table linking one human protein to the Pfam domain that is engaged
by the `DEG_Nend` family of motifs.  In a real analysis this file comes from
`download_human_domains`, which downloads Pfam annotations for all expressed human
proteins from UniProt.

The format is a tab-separated file with columns `Entry`, `Pfam`, and `Gene Names`.
This is the same format as the bacterial domain table used in the forward pipeline, so
we can read it with `read_bacterial_domain_table` from `microbiolink_api`.

| Entry  | Pfam    | Gene Names | Biological role |
|--------|---------|------------|-----------------|
| P99999 | PF02207 | UBR1       | N-recognin (UBR-box), recognises N-degron substrates |

In [4]:
bacterial_fasta_file = workdir / 'mock_bacterial_proteins.fasta'
human_domain_file    = workdir / 'mock_human_domains.tsv'

# A0A001: starts with MCR → triggers DEG_Nend_UBRbox_4 (^M{0,1}(C).)
# A0A002: contains RRKRG → triggers CLV_PCSK_FUR_1 (R.[RK]R.) but will be filtered
bacterial_fasta_file.write_text(
    '>sp|A0A001|DEG_BACT MimicProtein OS=Bacterium\n'
    'MCRAAAAAAAAAAAAAAAAAAAAAAAAAAA\n'
    '>sp|A0A002|CLV_BACT CleavageSiteProtein OS=Bacterium\n'
    'GGGRRRKRGGGGGGGGGGGGGGGGGGGGG\n',
    encoding='utf-8',
)

# Human protein P99999 carries the zf-UBR domain (PF02207)
human_domain_file.write_text(
    'Entry\tPfam\tGene Names\n'
    'P99999\tPF02207\tUBR1\n',
    encoding='utf-8',
)

print('Mock files written:')
print(' ', bacterial_fasta_file)
print(' ', human_domain_file)

Mock files written:
  /Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2/tutorials/mock_data/runs/03_mock_reverse_dmi/mock_bacterial_proteins.fasta
  /Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2/tutorials/mock_data/runs/03_mock_reverse_dmi/mock_human_domains.tsv


## Step 2: Load ELM resources and filter out CLV_ motifs

`load_default_dmi_resource_bundle()` reads two packaged files that ship with
`microbiolink_api`:

- **`elm_classes.tsv`** — a table of ELM motif identifiers and their regular
  expression patterns.  Each row defines one class of short linear motif, e.g.
  `DEG_Nend_UBRbox_4` with regex `^M{0,1}(C).`.
- **`elm_interaction_domains.tsv`** — a table linking ELM motif identifiers to the
  Pfam domain families that recognise them, e.g. `DEG_Nend_UBRbox_4 → PF02207
  (zf-UBR)`.

### Why CLV_ motifs must be removed

ELM motif identifiers are prefixed by a two- or three-letter code that describes the
motif's function:

| Prefix | Meaning | Example |
|--------|---------|---------|
| `LIG_` | Ligand-binding (domain interaction) | `LIG_SH2_STAT5` |
| `DEG_` | Degradation signal | `DEG_Nend_UBRbox_4` |
| `MOD_` | Post-translational modification site | `MOD_PKA_1` |
| `DOC_` | Docking site | `DOC_PP1_RVXF_1` |
| `CLV_` | **Cleavage site** | `CLV_PCSK_FUR_1` |

`CLV_` patterns mark sites where proteases cut the polypeptide chain.  A protease
recognises and *cleaves* its substrate — it does not form a stable domain-mediated
complex.  Using `CLV_` patterns in a domain-motif interaction screen would therefore
produce biologically incorrect predictions.  `filter_cleavage_motifs()` removes every
key whose name begins with `CLV_` from the regex dictionary before any sequence
scanning takes place.

In [5]:
resources = load_default_dmi_resource_bundle()

n_total   = len(resources.elm_regex)
n_clv     = sum(1 for name in resources.elm_regex if name.startswith('CLV_'))
print(f'ELM motif patterns loaded : {n_total}')
print(f'  of which CLV_ (cleavage) : {n_clv}')

filtered_elm_regex = filter_cleavage_motifs(resources.elm_regex)
n_after = len(filtered_elm_regex)
print(f'After CLV_ filtering      : {n_after} patterns remain')
print(f'Removed                   : {n_total - n_after} cleavage-site patterns')
print()
print('Motif-domain interaction pairs loaded:', len(resources.motif_domains))

ELM motif patterns loaded : 354
  of which CLV_ (cleavage) : 11
After CLV_ filtering      : 343 patterns remain
Removed                   : 11 cleavage-site patterns

Motif-domain interaction pairs loaded: 360


## Step 3: Load bacterial sequences and the human domain table

### Bacterial FASTA

`read_fasta_sequences()` returns a `dict` mapping each FASTA header (the line after
`>`, without the leading `>`-character) to the full protein sequence string.  We will
scan these sequences in Step 4.

### Human domain table

`read_bacterial_domain_table()` parses a two-column (or more) tab-separated file whose
first column is a UniProt accession and whose second column contains one or more
semicolon-separated Pfam identifiers.  It returns a `dict` mapping each Pfam domain
identifier to the list of proteins that carry it:

```python
{'PF02207': ['P99999'], ...}
```

This is exactly the format produced by `download_human_domains` for the expressed
human proteome.  Although the function is named `read_bacterial_domain_table`, the
file format is identical — we re-use it here for the human side of the interaction.

In [7]:
bacterial_sequences = read_fasta_sequences(bacterial_fasta_file)
human_domain_table  = read_bacterial_domain_table(human_domain_file)

print(f'Bacterial proteins loaded: {len(bacterial_sequences)}')
for header in bacterial_sequences:
    uid = extract_uniprot_id(header)
    seq = bacterial_sequences[header]
    print(f'  {uid}: {seq[:10]}... ({len(seq)} aa)')

print()
print(f'Human domain table: {len(human_domain_table)} Pfam domain(s) covered')
for domain, proteins in human_domain_table.items():
    print(f'  {domain}: {proteins}')

Bacterial proteins loaded: 2
  A0A001: MCRAAAAAAA... (30 aa)
  A0A002: GGGRRRKRGG... (29 aa)

Human domain table: 1 Pfam domain(s) covered
  PF02207: ['P99999']


## Step 4: Run the reverse domain-motif interaction matching

`predict_reverse_domain_motif_interactions_from_data()` runs the full three-stage
algorithm in a single call:

1. **Scan** each bacterial sequence for ELM motif hits using `re.finditer`.
2. **Look up** the Pfam domains that each matched motif is known to interact with.
3. **Look up** the human proteins annotated with each of those domains.

It returns a list of `ReverseDomainMotifInteraction` dataclasses, one entry per
`(bacterial protein, motif hit, human domain, human protein)` combination.

In [8]:
interactions = predict_reverse_domain_motif_interactions_from_data(
    bacterial_sequences=bacterial_sequences,
    elm_regex=filtered_elm_regex,
    motif_domains=resources.motif_domains,
    human_domain_table=human_domain_table,
)

print(f'Predicted {len(interactions)} reverse domain-motif interaction(s).')

Predicted 1 reverse domain-motif interaction(s).


In [9]:
interaction_frame = pd.DataFrame(
    [dataclasses.asdict(i) for i in interactions],
)

interaction_frame

,bacterial_protein,motif,start,end,human_domain,human_protein
0,A0A001,DEG_Nend_UBRbox_4,0,3,PF02207,P99999


## Step 5: Save the results

We write the interaction table in two formats:

- **Tab-separated (`.tsv`)** — for downstream analysis in Python, R, or a spreadsheet
  tool.
- **Semicolon-separated legacy (`.csv`)** — matches the output format expected by
  subsequent pipeline steps such as AIUPred structural filtering.

### Output column reference

| Column | Description |
|--------|-------------|
| `bacterial_protein` | UniProt accession of the bacterial protein carrying the motif |
| `motif` | ELM identifier of the matched motif class |
| `start` | 0-based index of the first matched residue in the bacterial sequence |
| `end` | 0-based index of the first residue *after* the match (Python slice convention) |
| `human_domain` | Pfam domain identifier known to bind this motif class |
| `human_protein` | UniProt accession of the human protein carrying that domain |

In [10]:
dmi_table_file  = workdir / 'reverse_dmi_output.tsv'
dmi_legacy_file = workdir / 'reverse_dmi_legacy.csv'

interaction_frame.to_csv(dmi_table_file,  sep='\t', index=False)
interaction_frame.to_csv(dmi_legacy_file, sep=';',  index=False, header=False)

print('Results written to:')
print(' ', dmi_table_file)
print(' ', dmi_legacy_file)

Results written to:
  /Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2/tutorials/mock_data/runs/03_mock_reverse_dmi/reverse_dmi_output.tsv
  /Users/tjl124/Documents/code_projects/microbiolink2_new/MicrobioLink2/tutorials/mock_data/runs/03_mock_reverse_dmi/reverse_dmi_legacy.csv


## Result

### What we expect to see

- **`A0A001`** appears in the output.  Its N-terminal `MCR` sequence triggers
  `DEG_Nend_UBRbox_4` (and possibly related N-degron classes), which interacts with
  `PF02207` (zf-UBR).  Human protein `P99999` carries `PF02207`, so a predicted
  interaction is reported.

- **`A0A002`** does **not** appear in the output.  Its `RRKRG` subsequence would
  match the furin cleavage regex (`CLV_PCSK_FUR_1`), but that pattern was removed by
  `filter_cleavage_motifs` before scanning.  Its sequence also starts with `G`, so
  none of the N-degron patterns (which are anchored to `^M{0,1}`) can fire.  Because
  our mock human domain table contains only `PF02207` entries, no other motif hit on
  `A0A002` can generate an output row.

### Connecting to a real analysis

In a full reverse MicrobioLink run you would replace the mock files with:

| Mock file | Real source |
|-----------|-------------|
| `mock_bacterial_proteins.fasta` | Output of `get_bacterial_fasta` (all proteins in the target microbiome's proteome) |
| `mock_human_domains.tsv` | Output of `download_human_domains` (Pfam annotations for expressed human proteins after z-score filtering) |

The ELM resources (`elm_classes.tsv` and `elm_interaction_domains.tsv`) are shared
between the forward and reverse pipelines and are loaded automatically from the
package by `load_default_dmi_resource_bundle()`.